# Data Pre-Processing - Extracting relevant information from TripAdvisor list pages

In [1]:
import os
from pathlib import Path
from bs4 import BeautifulSoup
import pandas as pd
import json
from collections import Counter

First, I read in the pages containing the list items from Tripadvisor.

In [2]:
files = os.listdir('tripadvisor_pages')
print(f"Total files: {len(files)}")

Total files: 98


Next, I extract relevant information. For a lot of parts, I have tested this in the previous notebook. Here I also retrieve the categories assigned to an attractions.

In [3]:
all_attractions = []

for offset in range(0, 73 * 30, 30):
    filepath = f'tripadvisor_pages/page_{offset}.html'
    
    if not os.path.exists(filepath):
        continue
    
    with open(filepath, 'r', encoding='utf-8') as f:
        soup = BeautifulSoup(f.read(), 'html.parser')
    
    review_tags = soup.find_all(attrs={'data-automation': 'bubbleReviewCount'})
    
    for review_tag in review_tags:
        # Name - strip ranking number
        name_tag = review_tag.find_previous('h3')
        if name_tag:
            rank_span = name_tag.find('span', class_='vAUKO')
            if rank_span:
                rank_span.decompose()
            name = name_tag.get_text(strip=True)
        else:
            name = None
        
        # Link
        link_tag = review_tag.find_previous('a', href=lambda h: h and 'Attraction_Review' in h)
        link = 'https://www.tripadvisor.com' + link_tag['href'] if link_tag else None
        
        # Rating
        rating_tag = review_tag.find_previous(attrs={'data-automation': 'bubbleRatingValue'})
        rating = rating_tag.get_text(strip=True) if rating_tag else None
        
        # Review count
        review_count = review_tag.get_text(strip=True).strip('()').replace(',', '')

        # Type Description
        category = None
        if name_tag:
            for div in name_tag.find_all_next('div', class_=lambda c: c and 'alPVI' in c and 'PgLKC' in c and 'tnGGX' in c):
                raw = div.find(string=True, recursive=False)
                if not raw:
                    first_child = div.find(True, recursive=False)
                    raw = first_child.get_text(strip=True) if first_child else None
                else:
                    raw = raw.strip()
                
                category = [c.strip() for c in raw.split('•')] if raw else None
                break

        all_attractions.append({
        'name': name,
        'link': link,
        'rating': rating,
        'review_count': review_count,
        'category': category,
        })
        

df_tripadvisor = pd.DataFrame(all_attractions)
print(f"Total attractions extracted: {len(df_tripadvisor)}")
print(df_tripadvisor.head(10))

Total attractions extracted: 2178
                                       name  \
0                           Hallgrimskirkja   
1                                    Perlan   
2                            Glacier Lagoon   
3                            Gullfoss Falls   
4                                Sky Lagoon   
5                                 Skogafoss   
6                               Blue Lagoon   
7  Harpa Concert Hall and Conference Centre   
8                 Thingvellir National Park   
9                  Seljalandsfoss waterfall   

                                                link rating review_count  \
0  https://www.tripadvisor.com/Attraction_Review-...    4.4        23266   
1  https://www.tripadvisor.com/Attraction_Review-...    4.5         4305   
2  https://www.tripadvisor.com/Attraction_Review-...    4.9         5082   
3  https://www.tripadvisor.com/Attraction_Review-...    4.7        12522   
4  https://www.tripadvisor.com/Attraction_Review-...    4.5         

I will save just to be sure that I have it in case things get lost.

In [4]:
records = df_tripadvisor.to_dict(orient='records')

with open('tripadvisor_attractions.json', 'w', encoding='utf-8') as f:
    json.dump(records, f, ensure_ascii=False, indent=2)

From browsing the website and the data, I can see that not all categories will be applicable for my routing app, so I will need to look at them and remove those I do not need.

In [5]:
category_counts = Counter(cat for cats in df_tripadvisor['category'] for cat in cats)
print(category_counts.most_common())

[('Multi-day Tours', 207), ('Speciality & Gift Shops', 183), ('Private Tours', 174), ('4WD, ATV & Off-Road Tours', 165), ('Sightseeing Tours', 142), ('Taxis & Shuttles', 122), ('Nature & Wildlife Tours', 94), ('Horseback Riding Tours', 76), ('Points of Interest & Landmarks', 75), ('City Tours', 75), ('Speciality Museums', 74), ('Waterfalls', 70), ('Geologic Formations', 69), ('Adrenaline & Extreme Tours', 64), ('Boat Tours', 62), ('Walking Tours', 58), ('Hiking & Camping Tours', 54), ('Day Trips', 51), ('Bars & Clubs', 50), ('Churches & Cathedrals', 49), ('Spas', 43), ('History Museums', 40), ('Cultural Tours', 39), ('Historic Sites', 38), ('Historical & Heritage Tours', 35), ('Eco Tours', 32), ('Nature & Wildlife Areas', 31), ('Monuments & Statues', 30), ('Ski & Snow Tours', 30), ('Art Galleries', 28), ('Scuba & Snorkelling', 27), ('Gear Rentals', 27), ('Thermal Spas', 22), ('Hot Springs & Geysers', 22), ('Sports Complexes', 21), ('Mountains', 21), ('Lighthouses', 21), ('Game & Entert

I manually went through all categories and decided which ones I should remove. For example, I do not want any tour companies.

In [6]:
remove_categories = {'Multi-day Tours', 'Private Tours', '4WD, ATV & Off-Road Tours', 'Sightseeing Tours',
                    'Taxis & Shuttles', 'Nature & Wildlife Tours', 'Horseback Riding Tours', 'City Tours',
                    'Adrenaline & Extreme Tours', 'Boat Tours', 'Walking Tours', 'Hiking & Camping Tours',
                    'Day Trips', 'Cultural Tours', 'Historical & Heritage Tours', 'Eco Tours', 'Ski & Snow Tours',
                    'Gear Rentals', 'Bus Tours', 'Climbing Tours', 'Helicopter Tours', 'Fishing Charters & Tours',
                    'Night Tours', 'Bike Tours', 'Photography Tours', 'Dolphin & Whale Watching', 
                    'Self-Guided Tours & Rentals', 'Bus Services', 'Beer Tastings & Tours', 'Bar, Club & Pub Tours',
                    'Canyoning & Rappelling Tours', 'Food Tours', 'Hop-On Hop-Off Tours', 'Safaris', 'Air Tours',
                    'Cooking Classes', 'Scavenger Hunts', 'Speed Boats Tours', 'Factory Tours', 'Motorcycle Tours',
                    'Wine Tours & Tastings', 'Distillery Tours', 'Swim with Dolphins', 'Shopping Tours', 'Movie & TV Tours',
                    'Segway Tours', 'Horse-Drawn Carriage Tours', 'Boat Rentals', 'Ghost & Vampire Tours', 'Duck Tours',
                    'Coffee & Tea Tours'} 

df_filtered = df_tripadvisor[
    ~df_tripadvisor['category'].apply(lambda cats: any(cat in remove_categories for cat in cats))]

In [7]:
category_counts_new = Counter(cat for cats in df_filtered['category'] for cat in cats)
print(category_counts_new.most_common())

[('Speciality & Gift Shops', 183), ('Points of Interest & Landmarks', 75), ('Speciality Museums', 74), ('Waterfalls', 70), ('Geologic Formations', 69), ('Bars & Clubs', 50), ('Churches & Cathedrals', 49), ('Spas', 43), ('History Museums', 40), ('Historic Sites', 38), ('Monuments & Statues', 30), ('Nature & Wildlife Areas', 30), ('Art Galleries', 28), ('Thermal Spas', 22), ('Hot Springs & Geysers', 22), ('Sports Complexes', 21), ('Mountains', 21), ('Lighthouses', 21), ('Game & Entertainment Centers', 21), ('Caverns & Caves', 19), ('Hiking Trails', 18), ('Art Museums', 17), ('Golf Courses', 16), ('Bodies of Water', 15), ('Beaches', 15), ('Water Parks', 15), ('Scenic Walking Areas', 14), ('Natural History Museums', 14), ('Farms', 13), ('Visitor Centers', 13), ('National Parks', 12), ('Lookouts', 12), ('Parks', 11), ('Scuba & Snorkelling', 11), ('Coffeehouses', 10), ('Architectural Buildings', 9), ('Religious Sites', 9), ('Canyons', 8), ('Kayaking & Canoeing', 8), ('Movie Theaters', 7), ('

In [8]:
len(df_filtered)

1112

Now I can export a dictionary with only the relevant attractions.

In [9]:
records_filtered = df_filtered.to_dict(orient='records')

with open('tripadvisor_attractions_filtered.json', 'w', encoding='utf-8') as f:
    json.dump(records_filtered, f, ensure_ascii=False, indent=2)